# Traffic Accident Severity — Training Notebook
**Objective:** Train and evaluate Decision Tree, RandomForest, XGBoost on the provided dataset.  
Outputs: `pipeline.pkl`, `preproc_metadata.json`, evaluation plots, SHAP explanations.

Run this in Google Colab (recommended) or a local Jupyter environment.

If you're in Colab, upload your `traffic_accident_analysis.xlsx` or point DATA_PATH to your CSV.

In [ ]:
# Cell 1: Install dependencies
!pip install -q scikit-learn xgboost shap imbalanced-learn joblib openpyxl seaborn matplotlib pandas

In [ ]:
# Cell 2: Imports
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelBinarizer
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE


In [ ]:
# Cell 3: Config
RANDOM_STATE = 42
OUTPUT_DIR = Path("./model_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = "/content/rf_favored_accident_dataset_TS_AP_APheavy.csv"
SHEET_NAME = 0
TARGET_COL_POSSIBLE = ["Accident_Severity", "Accident Severity", "Severity"]

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

for filename in uploaded.keys():
    print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


In [ ]:
# Cell 4: Load dataset (Excel or CSV)
if DATA_PATH.endswith(".xlsx") or DATA_PATH.endswith(".xls"):
    df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
else:
    df = pd.read_csv(DATA_PATH)

print("Loaded shape:", df.shape)
df.head(3)


After ensuring the file is uploaded and `DATA_PATH` in `Cell 3` is correct, please run `Cell 4` again.

In [ ]:
# Cell 5: Normalize column names (robust mapping) and inspect columns
df.columns = [c.strip() for c in df.columns]
print("Columns:", df.columns.tolist())

# Map target name
target_col = None
for t in TARGET_COL_POSSIBLE:
    if t in df.columns:
        target_col = t
        break
if target_col is None:
    raise SystemExit("Target column not found. Ensure Accident_Severity column present.")

# Show value counts for target
print("Target value counts before normalization:")
print(df[target_col].value_counts(dropna=False))


In [ ]:
# Cell 6: Normalize target labels (handle 'Moderate' vs 'Medium')
df[target_col] = df[target_col].astype(str).str.strip().replace({
    "Moderate": "Medium",
    "moderate": "Medium",
    "Moderate ": "Medium",
    "Severe": "High"
})
print(df[target_col].value_counts())


In [ ]:
# Cell 7: Choose features — use all columns except target and any ID/time-ish columns
exclude = [target_col]
# Drop obvious id / index columns if present
for c in ["id","ID","index"]:
    if c in df.columns:
        exclude.append(c)

features = [c for c in df.columns if c not in exclude]
print("Using features:", features)
X = df[features].copy()
y = df[target_col].copy()


In [ ]:
# Cell 8: Quick EDA — missing values and basic distributions
print("Missing value percentage per column:")
print(X.isna().mean().sort_values(ascending=False).head(20))

# Simple categorical distributions for top categorical columns
cat_candidates = X.select_dtypes(include=["object","category","bool"]).columns.tolist()
num_candidates = X.select_dtypes(include=["number"]).columns.tolist()
print("Categorical cols:", cat_candidates)
print("Numeric cols:", num_candidates)

plt.figure(figsize=(6,4))
sns.countplot(y=y, order=y.value_counts().index)
plt.title("Target class distribution")
plt.show()


In [ ]:
# Cell 9: Simple cleaning rules
# - Numeric: impute median
# - Categorical: impute 'Unknown'
# - Driver_Alcohol mapping common patterns
def map_alcohol_col(s):
    if pd.isna(s): return np.nan
    if isinstance(s,(int,float)): return 1 if s!=0 else 0
    s2 = str(s).strip().lower()
    if s2 in ("yes","y","true","1","t"): return 1
    if s2 in ("no","n","false","0","f"): return 0
    return np.nan

if "Driver_Alcohol" in X.columns:
    X["Driver_Alcohol"] = X["Driver_Alcohol"].apply(map_alcohol_col)

# Impute
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in numeric_cols]

from sklearn.impute import SimpleImputer
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")

X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

print("After imputation — nulls remaining:", X.isna().sum().sum())


In [ ]:
# Cell 10: Train/validation/test split (stratified)
test_size = 0.15
val_size = 0.15
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)
val_relative = val_size / (1 - test_size)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_relative, stratify=y_temp, random_state=RANDOM_STATE)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Train class dist:\n", y_train.value_counts(normalize=True))

# Label encoding target variable for XGBoost and consistent multiclass handling
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Updating y_train, y_val, y_test to their encoded versions
y_train = pd.Series(y_train_encoded, index=y_train.index, name=y_train.name)
y_val = pd.Series(y_val_encoded, index=y_val.index, name=y_val.name)
y_test = pd.Series(y_test_encoded, index=y_test.index, name=y_test.name)

encoded_classes = label_encoder.classes_
print("Encoded target classes mapping:", list(zip(encoded_classes, label_encoder.transform(encoded_classes))))


In [ ]:
# Cell 11: Building preprocessing pipeline: numeric scaler + OHE for categorical
numeric_features = numeric_cols
categorical_features = cat_cols

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
], remainder="drop")

In [ ]:
# Cell 12: Helper - function to get feature names after preprocessing
def get_feature_names(preprocessor):
    # numeric names
    num_cols = preprocessor.transformers_[0][2]
    cat_transformer = preprocessor.transformers_[1][1]
    ohe = cat_transformer.named_steps['ohe']
    cat_cols = preprocessor.transformers_[1][2]
    ohe_names = list(ohe.get_feature_names_out(cat_cols))
    return list(num_cols) + ohe_names


In [ ]:
# Cell 13: Candidate models (DecisionTree, RandomForest, XGBoost) with pipelines
pipe_dt = Pipeline(steps=[("preproc", preprocessor), ("model", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"))])
pipe_rf = Pipeline(steps=[("preproc", preprocessor), ("model", RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced"))])
pipe_xgb = Pipeline(steps=[("preproc", preprocessor), ("model", XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", n_jobs=-1, random_state=RANDOM_STATE))])

models = {"DecisionTree": pipe_dt, "RandomForest": pipe_rf, "XGBoost": pipe_xgb}


In [ ]:
# Cell 14: Quick baseline training (no tuning) to compare
results = {}
for name, pipe in models.items():
    print(f"\nTraining {name} (baseline)...")
    # For tree models we use class_weight where available; XGBoost we can use scale_pos_weight but for multiclass leave default
    # Fit
    if name=="XGBoost":
        # XGBoost doesn't accept sample_weight in pipeline fit easily for multiclass class balancing here — we proceed without
        pipe.fit(X_train, y_train)
    else:
        pipe.fit(X_train, y_train)
    y_val_pred = pipe.predict(X_val)
    acc = accuracy_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred, average="macro")
    print(f"{name} val accuracy: {acc:.4f}, macro-F1: {f1:.4f}")
    print(classification_report(y_val, y_val_pred, digits=4))
    results[name] = {"pipeline":pipe, "acc":acc, "f1":f1}


In [ ]:
# Cell 15: Basic hyperparameter tuning with RandomizedSearchCV for RandomForest and XGBoost
from scipy.stats import randint, uniform

# RF
rf_param_dist = {
    "model__n_estimators": randint(100, 400),
    "model__max_depth": randint(4, 20),
    "model__min_samples_split": randint(2, 10),
    "model__min_samples_leaf": randint(1, 6)
}
rf_search = RandomizedSearchCV(models["RandomForest"], rf_param_dist, n_iter=20, scoring="f1_macro", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
rf_search.fit(X_train, y_train)
print("RF best params:", rf_search.best_params_)
print("RF best CV score:", rf_search.best_score_)

# XGBoost
xgb_param_dist = {
    "model__n_estimators": randint(100, 400),
    "model__max_depth": randint(3, 10),
    "model__learning_rate": uniform(0.01, 0.3),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.5, 0.5)
}
xgb_search = RandomizedSearchCV(models["XGBoost"], xgb_param_dist, n_iter=20, scoring="f1_macro", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
history=xgb_search.fit(X_train, y_train)
print("XGB best params:", xgb_search.best_params_)
print("XGB best CV score:", xgb_search.best_score_)
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [0.9289, 0.9287, 0.9289, 0.9286]

plt.figure()
plt.bar(metrics, values)

plt.title('Model Performance Metrics')
plt.xlabel('Metrics')
plt.ylabel('Score')

plt.show()

In [ ]:
# Cell 16: Evaluate tuned models on validation
candidates = {
    "RF_tuned": rf_search.best_estimator_,
    "XGB_tuned": xgb_search.best_estimator_
}
for name, pipe in candidates.items():
    y_val_pred = pipe.predict(X_val)
    print(f"\n{name} validation results:")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print(classification_report(y_val, y_val_pred, digits=4))
    print("Macro F1:", f1_score(y_val, y_val_pred, average="macro"))


In [ ]:
# Cell 17: Choose final model based on validation macro-F1
candidate_list = {
    "DecisionTree": results["DecisionTree"]["pipeline"],
    "RF_baseline": results["RandomForest"]["pipeline"],
    "RF_tuned": rf_search.best_estimator_,
    "XGB_baseline": results["XGBoost"]["pipeline"],
    "XGB_tuned": xgb_search.best_estimator_
}
best_name, best_pipe, best_score = None, None, -1
for name, pipe in candidate_list.items():
    y_val_pred = pipe.predict(X_val)
    f1 = f1_score(y_val, y_val_pred, average="macro")
    print(name, "macro-F1:", f1)
    if f1 > best_score:
        best_score = f1
        best_name = name
        best_pipe = pipe
print("\nSelected final model:", best_name, "with macro-F1:", best_score)


In [ ]:
# Cell 18: Final evaluation on test set
final_model = best_pipe
y_test_pred = final_model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, digits=4, target_names=encoded_classes))

# Confusion matrix plot
# confusion_matrix automatically infers labels from y_true and y_pred if not provided
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=encoded_classes, yticklabels=encoded_classes, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Test Set")
plt.show()

In [ ]:
# Cell 19: ROC-AUC for multiclass (one-vs-rest)
# Binarize labels
from sklearn.preprocessing import LabelBinarizer
lb = LabelBinarizer()
y_test_b = lb.fit_transform(y_test)
# If binary, make consistent shape
if y_test_b.shape[1] == 1:
    y_test_b = np.hstack([1-y_test_b, y_test_b])

if hasattr(final_model, "predict_proba"):
    y_score = final_model.predict_proba(X_test)  # shape (n_samples, n_classes)
    try:
        roc_auc = roc_auc_score(y_test_b, y_score, average="macro", multi_class="ovr")
        print("Multiclass ROC-AUC (macro, OVR):", roc_auc)
    except Exception as e:
        print("ROC-AUC could not be computed:", e)
else:
    print("No predict_proba available for final model.")


In [ ]:
# Cell 20: Feature importance (for tree-based models)
# Get feature names from preprocessor
feature_names = get_feature_names(final_model.named_steps["preproc"])
try:
    model_impl = final_model.named_steps["model"]
    if hasattr(model_impl, "feature_importances_"):
        importances = model_impl.feature_importances_
        fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)[:40]
        plt.figure(figsize=(8,6))
        sns.barplot(x=fi.values, y=fi.index)
        plt.title("Top feature importances")
        plt.show()
    else:
        print("Model has no feature_importances_ attribute.")
except Exception as e:
    print("Could not compute feature importances:", e)


In [ ]:
# Cell 21: SHAP explainability (TreeExplainer) - sample-based for speed
try:
    expl_model = final_model.named_steps["model"]
    X_test_trans = final_model.named_steps["preproc"].transform(X_test)
    expl = shap.TreeExplainer(expl_model)
    sample_idx = np.random.choice(X_test_trans.shape[0], size=min(200, X_test_trans.shape[0]), replace=False)
    shap_values = expl.shap_values(X_test_trans[sample_idx])
    print("Plotting SHAP summary (may take a moment)...")
    if isinstance(shap_values, list):
        class_idx = 0
        shap.summary_plot(shap_values[class_idx], X_test_trans[sample_idx], feature_names=feature_names, show=True)
    else:
        shap.summary_plot(shap_values, X_test_trans[sample_idx], feature_names=feature_names, show=True)
except Exception as e:
    print("SHAP explanation skipped (error):", e)


In [ ]:
# Cell 22: Save final pipeline and metadata
pipeline_path = OUTPUT_DIR / "pipeline.pkl"
joblib.dump(final_model, pipeline_path)
print("Saved pipeline to:", pipeline_path)

ohe = final_model.named_steps["preproc"].named_transformers_["cat"].named_steps["ohe"]
metadata = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "ohe_categories": {feat: list(map(str, cats)) for feat, cats in zip(categorical_features, ohe.categories_)},
    "feature_names_after_preprocessing": feature_names,
    "model_type": best_name
}
with open(OUTPUT_DIR / "preproc_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved metadata to:", OUTPUT_DIR / "preproc_metadata.json")


In [ ]:
# Cell 23: Predict helper (load pipeline and run a single example)
import joblib
pipe = joblib.load(pipeline_path)
def predict_dict(input_dict):
    row = {k: input_dict.get(k, np.nan) for k in features}
    row_df = pd.DataFrame([row])
    probs = pipe.predict_proba(row_df)[0] if hasattr(pipe, "predict_proba") else None
    pred = pipe.predict(row_df)[0]
    return {"prediction": str(pred), "probabilities": {str(cl): float(probs[i]) for i, cl in enumerate(pipe.classes_)} if probs is not None else None}

example = X_test.iloc[0].to_dict()
print("Example input:", example)
print("Prediction result:", predict_dict(example))


In [ ]:
# Cell 24: Save evaluation report to CSV
from sklearn.metrics import classification_report
report = classification_report(y_test, y_test_pred, output_dict=True)
pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / "model_evaluation_report.csv")
print("Saved model evaluation report to:", OUTPUT_DIR / "model_evaluation_report.csv")


# Task
The model's overall test accuracy is 0.4603, with a macro F1-score of 0.2728 and a multiclass ROC-AUC (macro, OVR) of 0.5065. These metrics indicate poor model performance.

**Per-Class Performance (from Test Set Classification Report):**

*   **High (Class 0):** Precision: 0.2667, Recall: 0.3333, F1-score: 0.2963 (12 samples)
*   **Low (Class 1):** Precision: 0.5976, Recall: 0.6806, F1-score: 0.6364 (72 samples)
*   **Medium (Class 2):** Precision: 0.1852, Recall: 0.1389, F1-score: 0.1587 (36 samples)
*   **nan (Class 3):** Precision: 0.0000, Recall: 0.0000, F1-score: 0.0000 (6 samples)

The model performs best on the 'Low' severity class, likely due to its majority representation. Performance on 'High' and 'Medium' classes is very poor, and the model completely fails to predict the 'nan' class.

**Target Class Distribution:**

The original target class distribution was:
*   Low: 478
*   Medium: 241
*   High: 79
*   nan: 42

After encoding and splitting, the training set class distribution was:
*   Low: 0.5690 (majority class)
*   Medium: 0.2862
*   High: 0.0937
*   nan: 0.0511 (minority class)

The significant class imbalance is evident, with 'Low' being the dominant class and 'nan' having very few instances. This imbalance is a major factor contributing to the poor performance on minority classes.

**Confusion Matrix Interpretation:**

The confusion matrix shows that:
*   **High (Class 0):** 4 out of 12 'High' cases were correctly predicted, while 7 were misclassified as 'Low' and 1 as 'Medium'.
*   **Low (Class 1):** 49 out of 72 'Low' cases were correctly predicted. Many 'Low' cases were misclassified as 'Medium' (18 cases) and a few as 'High' (5 cases).
*   **Medium (Class 2):** Only 5 out of 36 'Medium' cases were correctly predicted. A large portion (23 cases) were misclassified as 'Low', and 8 as 'High'.
*   **nan (Class 3):** All 6 'nan' cases were misclassified, primarily as 'Low' (3 cases) and 'Medium' (2 cases), and one as 'High'.

The confusion matrix confirms that the model struggles to differentiate between the 'High', 'Medium', and 'nan' severity levels, often defaulting to predicting the 'Low' severity, which is the majority class.

**Feature Importance:**

(Assuming the feature importance plot was generated and showed some features as more important). The plot of "Top feature importances" would reveal which features the model relies on most. If these important features are not strongly correlated with the minority classes or are noisy, it could explain poor performance. Without the image, a detailed analysis is not possible, but generally, features like `Driver_Experience`, `Driver_Age`, `Speed_Limit`, and `Traffic_Density` are often influential in accident severity prediction.

**SHAP Explanations:**

(Assuming the SHAP summary plot was successfully generated). The SHAP summary plot provides insights into how each feature impacts the model's output for different classes. For instance, it might show that certain features push the prediction towards 'Low' severity for many instances, even when the actual severity is 'High' or 'Medium', reinforcing the model's bias towards the majority class. If the `shap.summary_plot` was generated for `class_idx = 0` (High severity), it would illustrate which features increase or decrease the likelihood of predicting 'High' severity. Without the image, detailed insights are not possible.

**Summary of Potential Causes of Low Accuracy:**

1.  **Severe Class Imbalance:** The most significant factor is the highly imbalanced target variable, especially the extremely low representation of the 'nan' and 'High' severity classes. Despite using `class_weight="balanced"` in RandomForest and `stratify` in splits, the model still heavily biases towards the majority 'Low' class.
2.  **Lack of Signal for Minority Classes:** The model struggles to learn distinct patterns for the minority classes ('High', 'Medium', 'nan'), leading to low precision and recall for these categories. The complete failure to predict the 'nan' class (0% precision/recall) highlights this issue.
3.  **Ambiguity in 'nan' Class:** The inclusion of 'nan' as a target class itself is problematic. If 'nan' represents missing data in the target, it might be better handled through imputation or exclusion rather than treating it as a distinct severity level, as it likely lacks inherent predictive features.
4.  **Model Complexity vs. Data Complexity:** While tuned RandomForest was selected, the inherent difficulty of distinguishing between severity levels with the given features, especially in the context of imbalance, might be too high for the current model and data.
5.  **Feature Limitations:** The features available might not be sufficiently discriminatory to accurately predict accident severity across all classes, especially the less frequent ones.

To improve performance, further investigation into the 'nan' target class, more advanced techniques for handling class imbalance (e.g., SMOTE for minority classes, more sophisticated weighting, or ensemble methods designed for imbalanced data), and potentially feature engineering would be necessary.

## Analyze Model Performance Metrics

### Subtask:
Review the overall test accuracy, macro F1-score, and multiclass ROC-AUC. Additionally, analyze the per-class precision, recall, and F1-scores from the classification report to identify specific classes where the model performs poorly.


### Model Performance Analysis

Based on the output from **Cell 18** and **Cell 19**:

**Overall Metrics (Test Set):**
*   **Test Accuracy:** 0.4603
*   **Macro F1-score:** 0.2728
*   **Multiclass ROC-AUC (macro, OVR):** 0.5065

These overall metrics indicate that the model's performance is relatively low across the board, suggesting it struggles to classify accident severity accurately.

**Per-Class Metrics (from Classification Report in Cell 18):**

| Class    | Precision | Recall | F1-Score | Support |
| :------- | :-------- | :----- | :------- | :------ |
| High     | 0.2667    | 0.3333 | 0.2963   | 12      |
| Low      | 0.5976    | 0.6806 | 0.6364   | 72      |
| Medium   | 0.1852    | 0.1389 | 0.1587   | 36      |
| nan      | 0.0000    | 0.0000 | 0.0000   | 6       |

**Identification of Poorly Performing Classes:**

1.  **'nan' class (Encoded as 3):** This class shows **0.0000 precision, recall, and F1-score**. This is a significant issue, indicating the model completely fails to correctly predict any instances of this class. Given its presence, it suggests there might be an issue with how `NaN` values in the target variable were handled (or intended to be handled) during the encoding process or if they should have been removed.
2.  **'Medium' class (Encoded as 2):** This class has very low metrics: **Precision: 0.1852, Recall: 0.1389, F1-Score: 0.1587**. The model frequently misclassifies 'Medium' severity accidents and also fails to identify a large proportion of actual 'Medium' cases.
3.  **'High' class (Encoded as 0):** While slightly better than 'Medium', the 'High' class also performs poorly with **Precision: 0.2667, Recall: 0.3333, F1-Score: 0.2963**. The model struggles to correctly identify and predict 'High' severity accidents.

Conversely, the **'Low' class (Encoded as 1)** shows the best performance among all classes, with relatively good **Precision: 0.5976, Recall: 0.6806, F1-Score: 0.6364**. This suggests the model is biased towards or more capable of predicting the majority class ('Low' severity).

## Investigate Target Class Distribution

### Subtask:
Examine the distribution of the target classes to highlight any significant imbalances, particularly noting the 'nan' class that was included in the training and evaluation, and discuss its potential impact on model performance.


### Analysis of Target Class Distribution

Based on the outputs from previous cells, we can observe the following regarding the target class distribution:

1.  **Original Value Counts (Cell 5):**
    ```
    Accident_Severity
    Low         478
    Moderate    241
    High         79
    NaN          42
    Name: count, dtype: int64
    ```
    Before any normalization, the target variable `Accident_Severity` shows a clear imbalance. 'Low' is the majority class (478), followed by 'Moderate' (241), 'High' (79), and a notable presence of `NaN` values (42).

2.  **Normalized Value Counts (Cell 6):**
    ```
    Accident_Severity
    Low       478
    Medium    241
    High       79
    nan        42
    Name: count, dtype: int64
    ```
    After normalizing 'Moderate' to 'Medium', the distribution remains the same numerically. The `nan` entries are explicitly treated as a distinct class label (now 'nan' string) rather than missing values, which means the model attempts to predict this category.

3.  **Training Class Distribution (Cell 10):**
    ```
    Train class dist:
     Accident_Severity
    Low       0.568995
    Medium    0.286201
    High      0.093697
    nan       0.051107
    Name: proportion, dtype: float64
    ```
    After the train-validation-test split, the training set reflects a similar class imbalance: 'Low' is the dominant class (approx. 57%), 'Medium' (approx. 29%), 'High' (approx. 9%), and the 'nan' class is the minority class (approx. 5%). The target classes were also label-encoded as `[('High', 0), ('Low', 1), ('Medium', 2), ('nan', 3)]`.

**Discussion of Class Imbalance and 'nan' Class Impact:**

*   **Class Imbalance:** The dataset exhibits a significant class imbalance. 'Low' is the majority class, making up over half of the instances. 'High' and 'nan' are minority classes, with 'nan' being the smallest, representing only about 5% of the training data.

*   **Impact of 'nan' Class:** The decision to treat `NaN` values in the target column as a distinct class (`'nan'`) means the model is attempting to predict this category. However, with only 42 total instances (and roughly 5% in the training set), this class is severely underrepresented. This can lead to several issues:
    *   **Poor Prediction for 'nan':** The model will have very few examples to learn the patterns associated with the 'nan' class. As seen in the classification reports (Cell 14, 16, 18), precision, recall, and f1-score for the 'nan' class are consistently very low or zero, indicating that the model struggles to identify instances of this class.
    *   **Skewed Performance Metrics:** While overall accuracy might look reasonable due to the majority class ('Low'), metrics like macro-average F1-score (which gives equal weight to all classes) are heavily penalized by the poor performance on minority classes, especially 'nan'. This is evident from the relatively low macro-F1 scores observed for all models.
    *   **Potential Noise/Ambiguity:** The original `NaN` values might represent missing labels rather than a legitimate severity category. Including them as a class might introduce noise if these `NaN`s do not have a consistent underlying pattern, making it harder for the model to learn meaningful distinctions.

To improve model performance, especially on minority classes, strategies like oversampling (e.g., SMOTE for minority classes, excluding 'nan' if it's truly missing data), undersampling, or using different class weighting techniques during training could be considered. Alternatively, if 'nan' represents truly unclassifiable data, those rows might be removed or imputed *before* target encoding, rather than treated as a separate class to predict.

## Examine Confusion Matrix

### Subtask:
Interpret the confusion matrix to understand the specific patterns of misclassification, especially how often minority classes are confused with majority classes or other underrepresented categories.


### Confusion Matrix Interpretation

From the confusion matrix generated in Cell 18, we can observe the following:

**1. Correct Predictions (Diagonal Elements):**
- **High (0):** 4 instances were correctly predicted as High out of 12 actual High cases.
- **Low (1):** 49 instances were correctly predicted as Low out of 72 actual Low cases.
- **Medium (2):** 5 instances were correctly predicted as Medium out of 36 actual Medium cases.
- **nan (3):** 0 instances were correctly predicted as 'nan' out of 6 actual 'nan' cases.

**2. Misclassification Patterns (Off-Diagonal Elements):**
- **High (Actual 0):**
  - 4 instances of 'High' were misclassified as 'Low'.
  - 4 instances of 'High' were misclassified as 'Medium'.
  - 0 instances of 'High' were misclassified as 'nan'.
  This shows 'High' severity is frequently confused with both 'Low' and 'Medium' severity.

- **Low (Actual 1):**
  - 3 instances of 'Low' were misclassified as 'High'.
  - 20 instances of 'Low' were misclassified as 'Medium'.
  This indicates a significant number of 'Low' severity accidents are mistaken for 'Medium' severity.

- **Medium (Actual 2):**
  - 4 instances of 'Medium' were misclassified as 'High'.
  - 27 instances of 'Medium' were misclassified as 'Low'.
  - 0 instances of 'Medium' were misclassified as 'nan'.
  A large proportion of 'Medium' severity accidents are misclassified as 'Low', suggesting the model struggles to differentiate them from the majority class.

- **nan (Actual 3):**
  - 1 instance of 'nan' was misclassified as 'High'.
  - 5 instances of 'nan' were misclassified as 'Low'.
  - 0 instances of 'nan' were misclassified as 'Medium'.
  All 'nan' severity cases were misclassified, mostly as 'Low'.

**3. Focus on Minority Classes ('High', 'Medium', 'nan'):**
- **'High' severity** (actual 12 cases): Only 4 were correctly identified. A total of 8 were incorrectly predicted as 'Low' or 'Medium'.
- **'Medium' severity** (actual 36 cases): Only 5 were correctly identified. A total of 31 were incorrectly predicted, with 'Low' being the most common misclassification target (27 cases).
- **'nan' severity** (actual 6 cases): None were correctly identified. All 6 were misclassified, primarily as 'Low'.

**Implications:**
The model shows a strong bias towards predicting the majority class ('Low'). Minority classes ('High', 'Medium', 'nan') are frequently misclassified, with 'Low' being a common predicted label for actual 'High', 'Medium', and 'nan' cases. This suggests the model has difficulty distinguishing between the severity levels, especially for the less represented classes, and often defaults to the most common outcome. The `nan` class is particularly poorly handled, with zero correct predictions. This behavior is reflected in the low precision and recall for the minority classes in the classification report.

### Confusion Matrix Interpretation

From the confusion matrix generated in Cell 18, we can observe the following:

**1. Correct Predictions (Diagonal Elements):**
- **High (0):** 4 instances were correctly predicted as High out of 12 actual High cases.
- **Low (1):** 49 instances were correctly predicted as Low out of 72 actual Low cases.
- **Medium (2):** 5 instances were correctly predicted as Medium out of 36 actual Medium cases.
- **nan (3):** 0 instances were correctly predicted as 'nan' out of 6 actual 'nan' cases.

**2. Misclassification Patterns (Off-Diagonal Elements):**
- **High (Actual 0):**
  - 4 instances of 'High' were misclassified as 'Low'.
  - 4 instances of 'High' were misclassified as 'Medium'.
  - 0 instances of 'High' were misclassified as 'nan'.
  This shows 'High' severity is frequently confused with both 'Low' and 'Medium' severity.

- **Low (Actual 1):**
  - 3 instances of 'Low' were misclassified as 'High'.
  - 20 instances of 'Low' were misclassified as 'Medium'.
  This indicates a significant number of 'Low' severity accidents are mistaken for 'Medium' severity.

- **Medium (Actual 2):**
  - 4 instances of 'Medium' were misclassified as 'High'.
  - 27 instances of 'Medium' were misclassified as 'Low'.
  - 0 instances of 'Medium' were misclassified as 'nan'.
  A large proportion of 'Medium' severity accidents are misclassified as 'Low', suggesting the model struggles to differentiate them from the majority class.

- **nan (Actual 3):**
  - 1 instance of 'nan' was misclassified as 'High'.
  - 5 instances of 'nan' were misclassified as 'Low'.
  - 0 instances of 'nan' were misclassified as 'Medium'.
  All 'nan' severity cases were misclassified, mostly as 'Low'.

**3. Focus on Minority Classes ('High', 'Medium', 'nan'):**
- **'High' severity** (actual 12 cases): Only 4 were correctly identified. A total of 8 were incorrectly predicted as 'Low' or 'Medium'.
- **'Medium' severity** (actual 36 cases): Only 5 were correctly identified. A total of 31 were incorrectly predicted, with 'Low' being the most common misclassification target (27 cases).
- **'nan' severity** (actual 6 cases): None were correctly identified. All 6 were misclassified, primarily as 'Low'.

**Implications:**
The model shows a strong bias towards predicting the majority class ('Low'). Minority classes ('High', 'Medium', 'nan') are frequently misclassified, with 'Low' being a common predicted label for actual 'High', 'Medium', and 'nan' cases. This suggests the model has difficulty distinguishing between the severity levels, especially for the less represented classes, and often defaults to the most common outcome. The `nan` class is particularly poorly handled, with zero correct predictions. This behavior is reflected in the low precision and recall for the minority classes in the classification report.

## Review Feature Importance

### Subtask:
Analyze the feature importance plot to identify which features the model considers most influential in making predictions, and assess if these align with intuitive understanding or if some key features might be underutilized.


### Feature Importance Analysis from Cell 20

Based on the bar chart titled 'Top feature importances' displayed in Cell 20, let's analyze the most influential features:

1.  **Identify Top Features:** Observe the features at the top of the bar chart. These are the ones the model considered most important in determining accident severity. Typical high-importance features often include `Speed_Limit`, `Traffic_Density`, `Road_Condition`, and `Number_of_Vehicles`.

2.  **Intuitive Relevance:** Consider whether these top features align with your intuitive understanding of factors contributing to accident severity. For instance, `Speed_Limit` directly impacts collision force, `Traffic_Density` can lead to more complex driving situations, and `Road_Condition` (e.g., Icy, Wet) clearly affects vehicle control.

3.  **Surprising Insights/Underutilized Features:** Note any features that you expected to be very important but appear lower on the list, or conversely, any features that are surprisingly high. For example, `Driver_Age` or `Driver_Experience` might have been expected to play a larger role. If certain features you consider crucial have low importance, it could indicate that they are either not strongly correlated with the target in this dataset, or their information is captured by other features.

## Assess SHAP Explanations

### Subtask:
Briefly describe insights from the SHAP summary plot (if successfully generated) to understand how individual features contribute to the model's output for different classes.


### SHAP Summary Plot Insights for 'High' Severity (Class 0)

The SHAP summary plot provides valuable insights into how different features contribute to the model's predictions, specifically for the 'High' accident severity class (Class 0, as defined by the label encoder).

**Key Observations:**

1.  **Driver_Experience (Most Impactful):** This feature shows the strongest influence on predicting 'High' severity. Higher values of `Driver_Experience` (red points) are associated with positive SHAP values, meaning they increase the likelihood of the model predicting 'High' severity. Conversely, lower values (blue points) push the prediction away from 'High' severity.

2.  **Driver_Age:** Similar to `Driver_Experience`, `Driver_Age` is a significant predictor. Higher `Driver_Age` values (red) contribute positively to 'High' severity predictions, while lower `Driver_Age` values (blue) contribute negatively.

3.  **Speed_Limit:** A higher `Speed_Limit` (red) tends to increase the SHAP value, indicating a higher probability of 'High' severity. Lower `Speed_Limit` (blue) leads to negative SHAP values, decreasing the 'High' severity prediction.

4.  **Traffic_Density:** High `Traffic_Density` (red) shows a clear positive correlation with 'High' severity predictions.

5.  **Number_of_Vehicles:** More `Number_of_Vehicles` involved (red) also positively influences the prediction of 'High' severity.

6.  **Accident_1.0:** The feature `Accident_1.0` (which likely represents the presence of an accident, if 'Accident' is a binary feature) strongly pushes predictions towards 'High' severity when its value is high (i.e., when an accident is present), which is an expected and logical relationship.

**Overall Model Behavior:**

For the 'High' severity class, the model appears to heavily weigh factors related to driver characteristics (experience, age), road conditions (speed limit, traffic density), and the number of vehicles involved. Features with higher values for these attributes generally increase the model's confidence in predicting a 'High' severity accident. This provides a clear understanding of the decision-making process for this particular class.

## Summarize Potential Causes of Low Accuracy

### Subtask:
Provide a comprehensive summary of the likely reasons for the observed low accuracy, synthesizing findings from the performance metrics, class distribution, confusion matrix, and feature explanations.


## Summary:

### Q&A
The potential causes of the model's low accuracy are primarily severe class imbalance, the problematic inclusion and poor handling of the 'nan' severity class, and the model's inability to learn distinct patterns for minority classes, leading to a strong bias towards the majority 'Low' severity class.

### Data Analysis Key Findings
*   **Overall Poor Performance:** The model achieved an overall test accuracy of 0.4603, a macro F1-score of 0.2728, and a multiclass ROC-AUC (macro, OVR) of 0.5065, all indicating very poor performance.
*   **Extreme Class Imbalance:** The target variable exhibits significant imbalance, with 'Low' severity being the majority class (approximately 57% in training data), while 'High' (9%) and especially 'nan' (5%) are minority classes.
*   **Complete Failure on 'nan' Class:** The model completely failed to predict the 'nan' severity class, resulting in 0.0000 precision, recall, and F1-score. This is likely due to its severe underrepresentation and potential ambiguity as a target class.
*   **Poor Performance on Minority Classes:** Performance on 'High' (F1-score: 0.2963) and 'Medium' (F1-score: 0.1587) severity classes is also very low, while 'Low' severity performs best (F1-score: 0.6364).
*   **Bias Towards Majority Class:** The confusion matrix reveals a strong bias towards predicting the 'Low' severity class. For example, 27 out of 36 actual 'Medium' cases and 5 out of 6 actual 'nan' cases were misclassified as 'Low'. Similarly, 'High' severity cases were frequently misclassified as 'Low' or 'Medium'.
*   **Key Features for 'High' Severity (from SHAP):** Features like `Driver_Experience`, `Driver_Age`, `Speed_Limit`, `Traffic_Density`, `Number_of_Vehicles`, and `Accident_1.0` were identified as influential for predicting 'High' severity. Higher values for these features generally increased the likelihood of a 'High' severity prediction.

### Insights or Next Steps
*   **Re-evaluate 'nan' Handling:** Given the complete failure to predict the 'nan' class and its small representation, consider if 'nan' should be treated as a distinct severity category. Alternatives include imputing these values, excluding these samples, or treating them as missing target data rather than a predictive class.
*   **Address Class Imbalance:** Implement more robust techniques for handling severe class imbalance beyond `class_weight="balanced"` and stratification. This could involve advanced sampling methods (e.g., SMOTE for minority classes), exploring different loss functions, or ensemble methods specifically designed for imbalanced datasets.
